In [ ]:
import os
import glob
import numpy as np
import xarray as xr

import matplotlib
from matplotlib import cm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1 import make_axes_locatable
%matplotlib inline

from joblib import Parallel, delayed

# --- gridded NetCDF + per-basin river profiles (PyGMT/ArcGIS + matplotlib) ---
from gospl.analyse.gridexport import (
    grid_export, to_netcdf, basin_rivers, plot_long_profile, plot_basin_map)

# --- stratigraphic sections / wells / Wheeler (matplotlib, inline) ---
from gospl.analyse.stratasection import (
    load_strata, cross_section, horizontal_slice, synthetic_well, wheeler,
    well_panel)

## Running the simulation

First activate the conda environment:

```bash
conda activate gospl
```

To run the simulation in a terminal (`X` = number of MPI processes, e.g. 5):

```bash
mpirun -np X gospl -i input-provenance.yml
```

# Analysing the outputs

All the post-processing below uses goSPL's built-in **`gospl.analyse`** toolkit
(imported in the first cell). Two complementary modules are used:

- **`gospl.analyse.gridexport`** — reassembles the unstructured mesh, rasterises
  every surface field of an output step onto a **regular grid**, runs D8
  hydrology (drainage area, basins, &chi;) and writes a CF-NetCDF for
  PyGMT/ArcGIS. It also exposes per-basin **river long-profile** helpers.
- **`gospl.analyse.stratasection`** — reads the recorded stratigraphy and draws
  **cross-sections, synthetic wells and Wheeler (chronostratigraphic)
  diagrams** (coloured by facies, lithology, provenance, &hellip;).

Every function used below has a **terminal equivalent** so the same products can
be generated outside Jupyter. These console commands &mdash; `gospl-grid`,
`gospl-section`, `gospl-strata-volume` &mdash; are installed with goSPL and are
shown in each section.

Each gridded NetCDF (one file per output step) holds, when available (every
variable carries its `units` and a `long_name` definition):

+ surface elevation `elev` (m) and the step's `sea_level` (m)
+ cumulative erosion/deposition `erodep` (m) and its rate `EDrate` (m/yr)
+ water / sediment fluxes `FA`, `fillFA`, `waterFill`, `sedLoad`
+ hydrology: `drainage_area`, `basin` id, `chi`, `flowdist`, and the
  priority-flood-`filled` elevation

In [ ]:
# Define output folder name for the simulation
out_path = 'results/'

if not os.path.exists(out_path):
    os.makedirs(out_path)

### Rasterising the outputs to a regular grid &mdash; `grid_export` / `to_netcdf`

`grid_export` reassembles the global mesh, interpolates a step's fields onto a
regular grid, runs the D8 hydrology and returns a dict of 2-D arrays;
`to_netcdf` writes that to a CF-NetCDF (each variable annotated with its `units`
and `long_name`). `getOutputs` below simply loops over the steps and writes one
`results/surface<step>.nc` per step.

**`grid_export(h5dir, mesh, step=None, ...)` &mdash; main options**

| Argument | Default | Meaning |
|---|---|---|
| `h5dir` | &ndash; | the run's `h5` output directory |
| `mesh` | &ndash; | global mesh `.npz` (vertices `v`, cells `c`) |
| `step` | last | output step to rasterise |
| `spacing` | median edge | grid resolution `dx[,dy]` (mesh units) |
| `fields` | all | subset of surface fields to include |
| `mn` | `0.5` | &chi; concavity `m/n` |
| `a0` | `1.0` | &chi; reference drainage area |
| `base_level` | run sea level | elevation defining the coast / outlets (catchment + &chi; datum) |
| `latlim` | `89` | (global meshes) crop the polar caps |

Global (spherical) meshes are auto-detected and gridded in lon/lat. The resolved
sea level is stored in each file (global attribute **and** a `sea_level`
variable), so the grid is self-describing.

**Terminal equivalent** (one step &rarr; one NetCDF):

```bash
gospl-grid --h5dir strati_provenance/h5 --mesh inputs/gospl_mesh.npz:v:c \
    --step 25 --spacing 500 --out results/surface25.nc
```

For the whole time series, loop in the shell:

```bash
for s in $(seq 0 25); do
  gospl-grid --h5dir strati_provenance/h5 --mesh inputs/gospl_mesh.npz:v:c \
      --step $s --spacing 500 --out results/surface$s.nc
done
```

In [ ]:
h5dir = "strati_provenance/h5"
mesh = "inputs/gospl_mesh.npz"
reso = 500

out_name = "surface"

def getOutputs(steps):

    # clear any stale .nc files first
    for f in glob.glob(os.path.join(out_path, f"{out_name}*.nc")):
        try:
            os.remove(f)
        except PermissionError:
            print(f"Still locked, close it first: {f}")
            return
        
    for stp in steps:
        g = grid_export(h5dir, mesh, stp, spacing=reso)
        fname = os.path.join(out_path, f"{out_name}{stp}.nc")
        to_netcdf(g, fname)                       

    return

def getOutputsParallel(steps, n_workers=8):
    for f in glob.glob(os.path.join(out_path, f"{out_name}*.nc")):
        try:
            os.remove(f)
        except PermissionError:
            print(f"Still locked, close it first: {f}")
            return

    def process_step(stp):
        g = grid_export(h5dir, mesh, stp, spacing=reso)
        fname = os.path.join(out_path, f"{out_name}{stp}.nc")
        to_netcdf(g, fname)

    Parallel(n_jobs=n_workers)(delayed(process_step)(stp) for stp in steps)

steps = np.arange(26)
getOutputsParallel(steps, n_workers=8)
# getOutputs(steps)

### Surface elevation through time

The four panels show the remapped `elevation` field at steps 5, 10, 15 and 25, with the black contour marking the $0$ m shoreline. Watch how the coastline migrates as the prescribed sea level and sediment supply reshape the margin: a seaward-stepping shoreline indicates progradation, a landward-stepping one indicates transgression.

In [ ]:
ncfiles = [os.path.join(out_path, f"{out_name}{stp}.nc") for stp in steps]
ds = {stp: xr.open_dataset(f) for stp, f in zip(steps, ncfiles)}
ds[5]

In [ ]:
stps = [5,10,15,25]
fig, axs = plt.subplots(2,2, figsize=(8,8), sharex=True, sharey=True)
for ax, stp in zip(axs.flat, stps):
    im = ds[stp].elev.plot(ax=ax, add_labels=False, add_colorbar=False, cmap='Spectral_r')
    ds[stp].elev.plot.contour(ax=ax, levels=[ds[stp].sea_level.values], colors=['k'])
for ax, stp in zip(axs.flat, stps):
    ax.set_title(f'step = {stp}', fontsize=10, fontweight="bold")
cbar_ax = fig.add_axes([0.2, -0.02, 0.6, 0.02]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Elevation (m)')
plt.show()

fig, axs = plt.subplots(2,2, figsize=(8,8), sharex=True, sharey=True)
for ax, stp in zip(axs.flat, stps):
    im = ds[stp].EDrate.plot(ax=ax, add_labels=False, add_colorbar=False, cmap='bwr')
    ds[stp].elev.plot.contour(ax=ax, levels=[ds[stp].sea_level.values], colors=['k'])
for ax, stp in zip(axs.flat, stps):
    ax.set_title(f'step = {stp}', fontsize=10, fontweight="bold")
cbar_ax = fig.add_axes([0.2, -0.02, 0.6, 0.02]) 
cbar = fig.colorbar(im, cax=cbar_ax, extend='both', orientation='horizontal')
cbar.set_label('Erosion / deposition rates (m/yr)')
plt.show()

### Longitudinal profile evolution

We collapse each step's grid to a mean profile along $y$ and overlay every second step in grey, with step 0 (red) and step 25 (blue) emphasised. The shaded region between the initial and final profiles shows the net change: where the final profile sits below the initial one the margin has aggraded/prograded a sediment wedge, illustrating how the depositional surface builds basinward over the $250$ kyr run.

In [ ]:
stp = 25
meands = ds[stp].mean(dim='y')
maxds = ds[stp].max(dim='y')
minds = ds[stp].min(dim='y')

plt.figure(figsize=(8,4))
ax = plt.gca()

meands.elev.plot(lw=2,c='k',label='mean')
maxds.elev.plot(lw=1,c='b',ls='-.',label='max')
minds.elev.plot(lw=1,c='r',ls='-.',label='min')

plt.xlabel('Distance x-axis (m)')
plt.ylabel('elevation (m)')
plt.title('Averaged longitudinal profile at the last time step', size=10)
plt.xlim(ds[stp].x.min(),ds[stp].x.max())
plt.legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
means = {stp: d.mean(dim='y') for stp, d in ds.items()}

fig, ax = plt.subplots(figsize=(8, 4))
cmap = plt.get_cmap('gray_r', len(ds) + 1)

for k in range(0, len(ds), 2):
    means[k].elev.plot(ax=ax, lw=1, ls='-.', c=cmap(k))

means[0].elev.plot(ax=ax, lw=2, c='r', label='step 0')
means[25].elev.plot(ax=ax, lw=2, c='b', label='step 25')

minz = np.minimum(means[25].elev, means[0].elev)
ax.fill_between(means[0].x, -520, minz, facecolor='gainsboro')

ax.set(
    xlabel='Distance x-axis (m)', ylabel='Elevation (m)',
    title='Averaged longitudinal profile through time',
    xlim=(means[0].x.min(), means[0].x.max()),
    ylim=(minz.min()-10, 330),
)
ax.legend(frameon=False, loc='upper right')
plt.tight_layout()
plt.show()

## Drainage basins and river long profiles

The gridded files already carry the `basin` ids and `chi`. To examine the
**channel network of a single basin**, `basin_rivers` traces the main stem (up
the largest-area donor at each step) and its tributaries; `plot_basin_map` maps
them with the **sea-level coastline**, and `plot_long_profile` draws the
longitudinal profile (distance in km).

**What you can do:** pick a basin (by id, or by clicking a point as below), set
the channel-defining drainage-area threshold, map the network over any gridded
field, and plot the long profile against distance or **&chi;** &mdash; either the
raw elevation (keeps real lakes as dips) or the hydrologically-`filled` one
(monotonic).

| Function | Key options | Meaning |
|---|---|---|
| `basin_rivers(result, ...)` | `basin_id`, `area_threshold` | basin to extract (default: largest); min drainage area (m&sup2;) for a channel (default: 95th pct) |
| `plot_basin_map(result, rivers, ...)` | `background`, `sea_level`, `figsize` | base field (default `elev`); coastline datum (default run sea level); figure size |
| `plot_long_profile(rivers, ...)` | `xaxis`, `which`, `figsize` | `dist` or `chi`; `elev` (raw) or `filled` (monotonic); figure size |

These per-basin plots are a **notebook API** (no dedicated console command); they
read the same grid `gospl-grid` produces. Below we first locate a basin id by
picking a point, then extract and plot its rivers.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

elev = ds[25].elev
elev.plot(ax=ax, cmap="gray", alpha=0.2, add_colorbar=False)

basin = ds[25].basin
basin.plot(ax=ax, cmap="jet",vmax=300)
levels = np.unique(ds[25].basin.values)
cs = ax.contour(basin.x, basin.y, basin.values, levels=levels, colors="k", linewidths=0.1)
cs = ax.contour(basin.x, basin.y, elev.values, levels=[ds[25].sea_level], colors="k", linestyles='-', linewidths=1)

plt.tight_layout()
plt.show()

In [ ]:
xbasin = 25.e3
ybasin = 43.e3
basin_id = int(ds[25].sel(x=xbasin, y=ybasin, method='nearest').basin.values)
print(f"Corresponding basin ID: {basin_id}")

In [ ]:
g = grid_export(h5dir, mesh, 25, spacing=reso)
riv = basin_rivers(g, basin_id=basin_id, area_threshold=5e6)
plot_basin_map(g, riv)
plot_long_profile(riv, which="elev")

# Extracting the stratigraphy

The recorded stratigraphy is stored as `HDF5` files in the output folder
(`stratal.<step>.p<rank>.h5`, where `<step>` is the output step and `<rank>` the
processor number).

### Stratal record

Each layer stores:

+ `stratZ` &mdash; elevation at the **time of deposition** (the current elevation
  for the top layer)
+ `stratH` &mdash; layer thickness (accounting for both erosion & deposition)
+ `phiS` &mdash; coarse-sediment porosity at the layer centre; with dual
  lithology also `stratHf` (fine thickness) / `phiF`, and with provenance the
  per-source-class `stratP`

`load_strata(h5dir, mesh, step=None, file_base="gospl")` reassembles the global
pile for one step and builds the **current interface elevations** (so eroded
layers pinch out). It also reads the step's **sea level** and **display time**
from the `.xmf`, which then serve as the defaults for the section / Wheeler
datum and the time axis.

**3-D volume for ParaView** &mdash; the whole pile can be turned into a wedge
(triangular-prism) volume with the `gospl-strata-volume` console command (run in
a terminal; `--field lithology` or `--field provenance`):

```bash
mpirun -np 4 gospl-strata-volume --h5dir strati_provenance/h5 --outdir outprov \
    --steps 25 --field provenance
```

### Cross-sections &mdash; `cross_section`

`cross_section` draws a vertical stratigraphic section along **x**, **y** or an
arbitrary **path**, with each layer coloured by a chosen property. The y-axis is
the **true elevation** (m); horizontal distance is shown in **km**. A light-grey
basement and a solid sea-level line are drawn behind the section.

**What you can do:** choose the transect (`kind`/`at`, or a `path` of waypoints);
colour by deposition elevation, thickness, lithology, coarse fraction, porosity,
age, provenance or **facies**; exaggerate the vertical (`vexag`); trace interface
lines every N layers (`layer_lines`); zoom with `xlim`/`ylim`; and restyle
(`figsize`, `cmap`, `legend_loc`, `title_fontsize`). It returns the matplotlib
`Axes`, so you can keep tuning it or `savefig` it.

`color_by="facies"` classifies every layer by the **water depth at the time of
deposition** (`sea_level - stratZ`); the breaks/colours/labels are tunable
(`facies_depths` = depth bin edges, m below sea level):

| Water depth at deposition | Facies (default) |
|---|---|
| above sea level | fluvial / deltaic plain |
| 0 &ndash; 20 m | shoreface |
| 20 &ndash; 50 m | distal offshore |
| 50 &ndash; 75 m | upper slope |
| > 75 m | lower slope |

**`cross_section(data, ...)` &mdash; options**

| Argument | Default | Meaning |
|---|---|---|
| `kind` / `at` / `path` | `'x'` | transect along x or y at position `at`, or an explicit `path` of (x,y) waypoints |
| `color_by` | `'lithology'` | `deposition` / `thickness` / `lithology` / `coarse` / `porosity` / `age` / `provenance` / `facies` |
| `vexag` | `1` | vertical exaggeration (true data aspect) &mdash; see note below |
| `sea_level` | from run | sea-level datum line **and** facies depth reference |
| `layer_lines` / `layer_line_kw` | `0` | thin interface line every N layers (+ a line-style dict) |
| `xlim` / `ylim` | full | distance / elevation range (m) &mdash; clips the view |
| `figsize` / `cmap` | &ndash; | figure size (w,h) / colormap (non-facies fields) |
| `legend_loc` / `title_fontsize` | &ndash; | facies-legend position / title font size |
| `facies_depths` / `facies_colors` / `facies_labels` | see above | tunable facies bins / colours / labels |
| `npts` | mesh res | number of samples along the transect |

> **`vexag` vs `xlim`/`ylim`:** with no pinned limits `vexag` locks a true
> vertical exaggeration while keeping the requested `figsize`. As soon as you set
> `xlim` and/or `ylim`, the view is clipped to exactly those bounds and the aspect
> is left auto (so `figsize` sets the proportions and `vexag` is ignored).

**Terminal equivalent:**

```bash
gospl-section --h5dir strati_provenance/h5 --mesh inputs/gospl_mesh.npz:v:c \
    --kind cross --step 25 --color-by facies --vexag 20 --figsize 8,5 \
    --xlim 0,200000 --ylim -400,300 --out section_facies.pdf
```

In [ ]:
data = load_strata(h5dir, mesh, step=25)

In [ ]:
# cross_section(data, kind="x", color_by="thickness", vexag=20, sea_level=0)
# cross_section(data, kind="x", color_by="porosity", vexag=20, sea_level=0)
fig, ax = plt.subplots(figsize=(10, 4))
cross_section(data, kind="x", color_by="facies", layer_lines=1, vexag=100,
              xlim=[0, 200.e3], ylim=[-400, 300], ax=ax)
# ax.figure.savefig("section_facies.pdf", dpi=200)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
cross_section(data, kind="x", color_by="thickness", layer_lines=1, vexag=100,
              xlim=[100.e3, 175.e3], ylim=[-400, 100], legend_loc=1, ax=ax)
# ax.figure.savefig("section_facies.pdf", dpi=200)

### Synthetic wells &mdash; `synthetic_well` & `well_panel`

`synthetic_well(data, x, y, ...)` draws the 1-D stratigraphic column at a point
as a borehole log coloured by any layer property. The colour bar is **horizontal
at the base** by default; a well is usually drawn **tall and narrow**
(`figsize=(1, 8)`).

**`synthetic_well(data, x, y, ...)` &mdash; options**

| Argument | Default | Meaning |
|---|---|---|
| `x`, `y` | &ndash; | well location (mesh coordinates, m) |
| `color_by` | `'lithology'` | `lithology` / `coarse` / `porosity` / `age` / `provenance` / `thickness` / `deposition` |
| `width` | `1.0` | column width (x-extent of the log) |
| `ylim` | well span | elevation range (m) |
| `vmin` / `vmax` | well's data range | colour-scale limits (set common values to compare wells) |
| `cbar_orientation` | `'horizontal'` | colour bar at the base, or `'vertical'` |
| `colorbar` | `True` | draw the per-well colour bar (set `False` when sharing one) |
| `cmap` / `figsize` / `title_fontsize` | &ndash; | colormap / figure size / title font size |

**Several wells on one figure** &mdash; `well_panel(data, locations, ...)` lays a
list of `(x, y)` wells side by side with a **shared colour scale** and a single
colour bar:

| Argument | Default | Meaning |
|---|---|---|
| `locations` | &ndash; | list of `(x, y)` well positions |
| `color_by` | `'lithology'` | property coloured in every well |
| `labels` | `"well 1"`, `"well 2"`, &hellip; | per-well titles |
| `vmin` / `vmax` | field range across the wells | shared colour-scale limits |
| `ylim` | union of the wells' elevation bounds | elevation range applied to every well |
| `figsize` / `sharey` / `title_fontsize` | &ndash; | figure size / share the y-axis / title font size |

**Terminal equivalent (single well):**

```bash
gospl-section --h5dir strati_provenance/h5 --mesh inputs/gospl_mesh.npz:v:c \
    --kind well --step 25 --xy 135000,50000 --color-by porosity \
    --figsize 1,8 --out well.pdf
```

In [ ]:
fig, ax = plt.subplots(figsize=(0.7, 8))
ax = synthetic_well(data, 135e3, 50e3, color_by="porosity", cbar_orientation='vertical', ax=ax)
# ax.figure.savefig("well.pdf", dpi=200)

In [ ]:
fig, axes = well_panel(data, [(115e3, 50e3), (135e3, 50e3), (145e3, 50e3)],
                       title_fontsize='9', ylim=[-280,-30], 
                       color_by="thickness", figsize=(2.1, 8))

### Wheeler (chronostratigraphic) diagram &mdash; `wheeler`

`wheeler(data, ...)` plots distance along a transect (x, in **km**) versus
**deposition time** (y, in **ky**); hiatuses and erosion are left **blank**, and
the **shoreline trajectory** is overlaid &mdash; the position where the
paleo-surface crossed sea level, drawn for **every** time step (from the recorded
`stratZ`, not only where deposits are preserved).

**What you can do:** read transgression/regression from the shoreline curve and
the blanks; pick the transect (`kind`/`at` or `path`); colour by thickness,
lithology, porosity, provenance, **facies**, &hellip;; set the time interval
(`dt`); and zoom with `xlim`/`ylim`.

| Argument | Default | Meaning |
|---|---|---|
| `kind` / `at` / `path` | `'x'` | transect along x or y at position `at`, or an explicit `path` of (x,y) |
| `color_by` | `'thickness'` | `thickness` / `deposition` / `lithology` / `coarse` / `porosity` / `age` / `provenance` / `facies` |
| `sea_level` | from run | shoreline-trajectory datum (and facies depth reference) |
| `dt` | from run | layer time interval (yr); default derived from the step's display time |
| `xlim` / `ylim` | full | distance (m) / deposition-time (yr) range |
| `facies_depths` / `facies_colors` / `facies_labels` | see cross-section | tunable facies bins / colours / labels |
| `figsize` / `legend_loc` / `title_fontsize` | &ndash; | figure size / legend position / title font size |

**Terminal equivalent:**

```bash
gospl-section --h5dir strati_provenance/h5 --mesh inputs/gospl_mesh.npz:v:c \
    --kind wheeler --step 25 --along x --at 20000 --color-by facies \
    --strat-dt 5000 --out wheeler.pdf
```

In [ ]:
fig, ax = plt.subplots(figsize=(4, 6))
ax = wheeler(data, kind="x", at=25000, color_by="facies", xlim=[100.e3,150.e3], legend_loc=4, dt=5e3, ax=ax)
# ax.figure.savefig("wheeler.pdf", dpi=200)